In [0]:
from pyspark.sql.functions import *

In [0]:
dim_date = spark.table("02_dev_silver.facts_and_dims.dim_date")
dim_customer = spark.table("02_dev_silver.facts_and_dims.dim_customer")
dim_product = spark.table("02_dev_silver.facts_and_dims.dim_product")
fact_order_items = spark.table("02_dev_silver.facts_and_dims.fact_order_items")

In [0]:
dim_date1=dim_date
fact_order_items = fact_order_items.join(
    dim_date1.select("date_key", "full_date"),
    fact_order_items.date_key == dim_date1.date_key,
    "left"
).select(
    "order_item_id",
    "order_id",
    "customer_key",
    "product_key",
    "quantity",
    "base_unit_price",
    "base_line_total",
    "order_status",
    col("full_date").alias("order_date"),
    "order_channel"
)
dim_customer = dim_customer.join(
    dim_date.select("date_key", "full_date"),
    dim_customer.registration_date_key == dim_date.date_key,
    "left"
).select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email_address",
    "customer_country",
    col("channel").alias("registration_channel"),
    col("full_date").alias("registration_date")
)
fact_order_items = fact_order_items.join(
    dim_customer,
    fact_order_items.customer_key == dim_customer.customer_key,
    "left"
).select(
    "order_item_id",
    "order_id",
    "product_key",
    "customer_id",
    "customer_name",
    "email_address",
    "customer_country",
    "registration_channel",
    "registration_date",
    "quantity",
    "base_unit_price",
    "base_line_total",
    "order_status",
    "order_date",
    "order_channel"
)
fact_order_items = fact_order_items.join(
    dim_product,
    fact_order_items.product_key == dim_product.product_key,
    "left"
).select(
    "order_item_id",
    "order_id",
    col("base_line_total").alias("total_amount"),
    "order_status",
    "order_date",
    "order_channel",
    "product_id",
    "product_name",
    "category",
    "price",
    "quantity",
    "currency",
    "exchange_rate_to_usd",
    "base_unit_price",
    "order_country",
    "customer_id",
    "customer_name",
    "email_address",
    "customer_country",
    "registration_channel",
    "registration_date"
)

In [0]:
fact_order_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("03_dev_gold.kpi_models.data_cube")